# 04 — Is the gap real, and what explains it?

The analysis showed a typical equity drag of ~1.4 pp/yr. Two honest follow-ups a
careful reviewer (or interviewer) would demand:

1. **Is the gap statistically real**, or could it be sampling noise? → a *paired*
   hypothesis test, because each fund gives us a matched Direct/Regular observation.
2. **What explains its size?** → a regression of drag on fund characteristics.

We close with a frank discussion of what this method can and cannot claim.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.formula.api as smf

from src import paths, plotting
plotting.set_style()

df = pd.read_csv(paths.RETURNS_CSV)
print(f"{len(df):,} funds")

1,084 funds


## 1. Paired hypothesis test

**Design.** For each fund we have two returns on the *same* portfolio — the Direct
plan and the Regular plan. That's a classic **paired** setup, so we test the
within-fund difference rather than comparing two independent groups.

- **H0:** the Direct and Regular plans deliver the same return (median difference = 0).
- **H1:** Direct ≠ Regular.

We use the **Wilcoxon signed-rank test** (non-parametric — it doesn't assume the
differences are normal, which matters given the artefact tails we found). We report
the parametric paired t-test alongside it as a cross-check, plus a bootstrap
confidence interval for the median drag so the result is an *effect size with
uncertainty*, not just a p-value.

In [2]:
d = df["direct_cagr"].values
r = df["regular_cagr"].values

w_stat, w_p = stats.wilcoxon(d, r)
t_stat, t_p = stats.ttest_rel(d, r)
n_direct_wins = int((d > r).sum())

print("Paired Wilcoxon signed-rank test")
print(f"  statistic = {w_stat:,.0f}   p-value = {w_p:.2e}")
print("Paired t-test (cross-check)")
print(f"  t = {t_stat:.1f}            p-value = {t_p:.2e}")
print(f"\nDirect plan outperformed in {n_direct_wins}/{len(df)} funds "
      f"({n_direct_wins/len(df)*100:.1f}%)")

Paired Wilcoxon signed-rank test
  statistic = 1,333   p-value = 2.58e-176
Paired t-test (cross-check)
  t = 29.7            p-value = 3.36e-142

Direct plan outperformed in 1073/1084 funds (99.0%)


In [3]:
# Bootstrap 95% CI for the median drag (effect size with uncertainty).
rng = np.random.default_rng(42)
drag = df["drag_pp"].values
boot = np.array([np.median(rng.choice(drag, size=len(drag), replace=True))
                 for _ in range(5000)])
lo, hi = np.percentile(boot, [2.5, 97.5])
print(f"Median drag (all funds): {np.median(drag):.2f} pp/yr")
print(f"95% bootstrap CI       : [{lo:.2f}, {hi:.2f}] pp/yr")

eq = df[df.broad_category == "Equity"]["drag_pp"].values
boot_eq = np.array([np.median(rng.choice(eq, size=len(eq), replace=True))
                    for _ in range(5000)])
lo_e, hi_e = np.percentile(boot_eq, [2.5, 97.5])
print(f"\nMedian drag (equity)   : {np.median(eq):.2f} pp/yr")
print(f"95% bootstrap CI       : [{lo_e:.2f}, {hi_e:.2f}] pp/yr")

Median drag (all funds): 0.81 pp/yr
95% bootstrap CI       : [0.77, 0.86] pp/yr



Median drag (equity)   : 1.44 pp/yr
95% bootstrap CI       : [1.40, 1.52] pp/yr


**Result.** The p-values are vanishingly small and the bootstrap CI for the
median drag sits comfortably above zero and well away from it. The gap is **not
noise** — it is a robust, systematic feature of the data. Note the test is almost
*too* easy to pass here: because the two plans hold the same portfolio, any
persistent fee difference shows up cleanly, which is precisely why this dataset is
such a clean natural experiment. The interesting quantity is the **effect size**
(~0.8 pp/yr overall, ~1.4 pp/yr equity), not the significance.

## 2. What explains the size of the drag?

We regress the per-fund drag on the characteristics we can observe: **broad
category** and the **length of the comparison window** (a proxy for how long the
fund has been around). We use the winsorised drag from nb 02 so a few NAV-artefact
outliers don't dominate the fit.

A caveat stated up front: the AMFI feed doesn't expose fund **AUM** or the exact
**expense ratio**, so this is a deliberately modest model. We're not trying to
*predict* drag for its own sake — the drag essentially *is* the commission. The
regression's job is to **localise** where the cost concentrates and quantify how
much each factor moves it.

In [4]:
model = smf.ols(
    "drag_pp_winsor ~ C(broad_category, Treatment(reference='Debt')) + window_years",
    data=df).fit()
print(model.summary().tables[1])
print(f"\nR-squared: {model.rsquared:.3f}  |  N = {int(model.nobs)}")

                                                                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------------------------------------------------
Intercept                                                         0.7113      0.036     19.978      0.000       0.641       0.781
C(broad_category, Treatment(reference='Debt'))[T.Equity]          0.8965      0.030     29.591      0.000       0.837       0.956
C(broad_category, Treatment(reference='Debt'))[T.Hybrid]          0.7089      0.038     18.433      0.000       0.633       0.784
C(broad_category, Treatment(reference='Debt'))[T.Other/Index]    -0.0833      0.034     -2.469      0.014      -0.150      -0.017
window_years                                                     -0.0237      0.003     -8.075      0.000      -0.029      -0.018

R-squared: 0.588  |  N = 1084


In [5]:
# Translate the category coefficients into plain predicted-drag terms.
pred = (df.assign(fitted=model.fittedvalues)
        .groupby("broad_category")["fitted"].median()
        .sort_values(ascending=False))
print("Model-implied typical drag by category (pp/yr):")
print(pred.round(2))

Model-implied typical drag by category (pp/yr):
broad_category
Equity         1.43
Hybrid         1.25
Other/Index    0.54
Debt           0.49
Name: fitted, dtype: float64


**Reading the model.**

- **Category is the dominant lever.** Relative to debt funds, **equity** funds carry
  a large, highly significant extra drag, with hybrids in between — mirroring the
  regulatory commission structure (equity plans can pay distributors more).
- **Window length** has only a small effect: the drag is a roughly steady annual
  leak, not something that systematically grows or shrinks with the measurement
  horizon — consistent with the "flat toll" finding in nb 03.
- **Category alone explains most of the variation** (R² ≈ 0.59 with just category
  + window length — high for so few, coarse features). That's the real takeaway:
  the drag isn't idiosyncratic fund-by-fund randomness, it's a *structural* cost
  that tracks the product type. What's left unexplained is mostly tracking noise
  and per-fund commission differences we can't see without TER data.

## 3. Limitations & honest caveats

A measurement is only as good as the assumptions behind it. The important ones:

1. **Survivorship bias (likely *understates* the problem).** We can only pair funds
   that are alive *today* with both plans. Funds that were merged away or shut down
   — disproportionately poor performers — are absent. If anything, the true
   long-run drag is probably *worse* than what we measure.
2. **Tracking noise over finite windows.** The two plans hold identical portfolios,
   but NAVs are rounded and occasionally stale, so over short windows the measured
   gap is noisy. We mitigated this with a ≥2-year floor and by reporting the median;
   the residual noise is why a handful of funds show impossible negative drag.
3. **What the gap does and doesn't include.** The realized return difference closely
   tracks the **expense-ratio (commission) difference**, but also folds in tiny
   cash-flow/tracking differences. Exit loads and taxes are *identical* across the
   two plans, so they don't bias the gap.
4. **No AUM / exact TER in the free feed.** A richer dataset (e.g. AMFI's TER
   disclosures or AMC factsheets) would let us model the commission directly and
   add fund-size effects. That's a clear next step.
5. **Distributor value isn't zero for everyone.** Some investors genuinely get
   planning/advice through a Regular plan, and the commission is the price of that.
   Our point is narrower and still holds: for the large number of investors who buy
   Regular plans *by default*, with no advice attached, the drag is pure leakage —
   and a free, identical Direct alternative exists.
6. **Realized, not guaranteed.** Numbers are historical (2013–present). The
   *mechanism* (Direct = Regular minus commission) is structural and forward-looking;
   the exact rupee figures depend on future returns.

## Bottom line

The Regular-vs-Direct gap is **statistically ironclad, economically large, and
concentrated in exactly the funds (equity) where Indians invest for the long term**.
It is one of the most cleanly measurable — and most ignored — leaks in Indian
personal finance, and closing it costs the investor nothing.